# Credit Risk — EDA & Model Comparison

Predicting loan default using machine learning.

**Dataset:** 2,000 synthetic applicants with realistic credit distributions  
**Target:** `default` (1 = defaulted, 0 = paid)

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from data.generate import generate

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

df = generate()
df.to_csv('../data/credit_risk.csv', index=False)
print(f'Shape: {df.shape}')
df.head()

## 1. Dataset Overview

In [ ]:
print('--- Info ---')
print(df.dtypes)
print('\n--- Missing values ---')
print(df.isnull().sum())
print('\n--- Target distribution ---')
print(df['default'].value_counts(normalize=True).round(3))

In [ ]:
df.describe().round(2)

## 2. Exploratory Data Analysis

In [ ]:
# Target balance
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

counts = df['default'].value_counts()
axes[0].pie(counts, labels=['No Default', 'Default'], autopct='%1.1f%%',
            colors=['#22c55e', '#ef4444'], startangle=90)
axes[0].set_title('Target Distribution')

sns.histplot(data=df, x='annual_income', hue='default', bins=30, ax=axes[1],
             palette={0: '#22c55e', 1: '#ef4444'}, alpha=0.7)
axes[1].set_title('Annual Income by Default Status')
axes[1].set_xlabel('Annual Income ($)')
plt.tight_layout()
plt.show()

In [ ]:
# Numeric distributions by default
num_cols = ['age', 'annual_income', 'loan_amount', 'debt_to_income_ratio', 'employment_years']
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(data=df, x='default', y=col, ax=axes[i],
                palette={0: '#22c55e', 1: '#ef4444'})
    axes[i].set_xticklabels(['No Default', 'Default'])
    axes[i].set_title(col.replace('_', ' ').title())
    axes[i].set_xlabel('')

axes[-1].set_visible(False)
plt.suptitle('Feature Distributions by Default Status', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Default rate by categorical features
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, ['credit_history', 'loan_purpose']):
    rates = df.groupby(col)['default'].mean().sort_values(ascending=False)
    colors = ['#ef4444' if r > 0.4 else '#f97316' if r > 0.25 else '#22c55e' for r in rates]
    rates.plot(kind='bar', ax=ax, color=colors, edgecolor='white', width=0.6)
    ax.set_title(f'Default Rate by {col.replace("_", " ").title()}')
    ax.set_ylabel('Default Rate')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(9, 7))
corr = df[num_cols + ['default']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn_r',
            center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
# Monthly payment burden
df['monthly_payment'] = (df['loan_amount'] / df['loan_duration_months']).round(2)
df['payment_to_income'] = (df['monthly_payment'] / (df['annual_income'] / 12)).round(3)

# Age group
df['age_group'] = pd.cut(df['age'], bins=[18, 30, 45, 60, 75],
                          labels=['18-30', '31-45', '46-60', '61-75'])

# Income bracket
df['income_bracket'] = pd.qcut(df['annual_income'], q=4,
                                 labels=['Q1', 'Q2', 'Q3', 'Q4'])

print('New features added:')
print(df[['monthly_payment', 'payment_to_income', 'age_group', 'income_bracket']].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

age_rates = df.groupby('age_group', observed=True)['default'].mean()
age_rates.plot(kind='bar', ax=axes[0], color='#3b82f6', edgecolor='white', width=0.6)
axes[0].set_title('Default Rate by Age Group')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

inc_rates = df.groupby('income_bracket', observed=True)['default'].mean()
inc_rates.plot(kind='bar', ax=axes[1], color='#8b5cf6', edgecolor='white', width=0.6)
axes[1].set_title('Default Rate by Income Bracket')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

plt.tight_layout()
plt.show()

## 4. Model Comparison

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, roc_curve
import warnings
warnings.filterwarnings('ignore')

NUMERIC = ['age', 'annual_income', 'employment_years', 'loan_amount',
           'loan_duration_months', 'num_credit_lines', 'debt_to_income_ratio']
CATEGORICAL = ['loan_purpose', 'credit_history']

X = df[NUMERIC + CATEGORICAL]
y = df['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

def make_pipeline(clf):
    pre = ColumnTransformer([
        ('num', StandardScaler(), NUMERIC),
        ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL),
    ])
    return Pipeline([('pre', pre), ('clf', clf)])

models = {
    'Logistic Regression': make_pipeline(LogisticRegression(max_iter=1000, random_state=42)),
    'Random Forest': make_pipeline(RandomForestClassifier(n_estimators=200, random_state=42)),
    'Gradient Boosting': make_pipeline(GradientBoostingClassifier(n_estimators=200, random_state=42)),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, pipe in models.items():
    cv_auc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc').mean()
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    results[name] = {
        'pipe': pipe,
        'cv_auc': cv_auc,
        'test_auc': roc_auc_score(y_test, y_proba),
        'accuracy': accuracy_score(y_test, y_pred),
        'proba': y_proba,
    }
    print(f'{name:25s} | CV AUC: {cv_auc:.4f} | Test AUC: {results[name]["test_auc"]:.4f} | Acc: {results[name]["accuracy"]:.4f}')

In [ ]:
# ROC curves
plt.figure(figsize=(8, 6))
colors = ['#3b82f6', '#22c55e', '#f97316']

for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['proba'])
    plt.plot(fpr, tpr, color=color, lw=2,
             label=f"{name} (AUC = {res['test_auc']:.3f})")

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Model Comparison')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Metrics comparison bar chart
metrics_df = pd.DataFrame({
    name: {'CV AUC': res['cv_auc'], 'Test AUC': res['test_auc'], 'Accuracy': res['accuracy']}
    for name, res in results.items()
}).T

ax = metrics_df.plot(kind='bar', figsize=(10, 5), width=0.7,
                     color=['#3b82f6', '#22c55e', '#8b5cf6'], edgecolor='white')
ax.set_title('Model Comparison — Metrics')
ax.set_ylabel('Score')
ax.set_ylim(0.5, 1.0)
ax.tick_params(axis='x', rotation=0)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (best tree model)
best_name = max(results, key=lambda k: results[k]['test_auc'])
best_pipe = results[best_name]['pipe']
clf = best_pipe.named_steps['clf']

if hasattr(clf, 'feature_importances_'):
    pre = best_pipe.named_steps['pre']
    cat_names = pre.named_transformers_['cat'].get_feature_names_out(CATEGORICAL).tolist()
    all_names = NUMERIC + cat_names
    fi = pd.Series(clf.feature_importances_, index=all_names).sort_values(ascending=True).tail(12)

    fi.plot(kind='barh', figsize=(9, 5), color='#3b82f6', edgecolor='white')
    plt.title(f'Feature Importance — {best_name}')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()

print(f'\nBest model: {best_name}')
print(f'Test AUC: {results[best_name]["test_auc"]:.4f}')

## 5. Conclusions

- **Best model:** Gradient Boosting (highest AUC)
- **Most important features:** `debt_to_income_ratio`, `annual_income`, `loan_amount`, `credit_history`
- **Key insight:** Credit history and debt-to-income ratio are the strongest predictors of default
- **Next steps:** Hyperparameter tuning with GridSearchCV, SHAP values for explainability